In [1]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks

# Obtener la ruta del directorio raíz del proyecto
project_root = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))

# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(project_root)

from config import INPUTS_FOLDER

statement_path = os.path.join(INPUTS_FOLDER, 'test_files', 'BBVA', '2206.pdf')

### Pipeline description

In [2]:
from document_reader import PDFReader
from bank_detector import DefaultBankDetector
import logging
logging.getLogger('pdfminer').setLevel(logging.ERROR) # Abvoid PDFMiner warnings

bank_detector = DefaultBankDetector(PDFReader(statement_path))

extracted_words = bank_detector.get_extracted_words()
extracted_words

,page,text,x0,top,x1,bottom
0,1,Estado,510.546000,1.219000,544.314000,13.219000
1,1,de,546.858000,1.219000,559.638000,13.219000
2,1,Cuenta,562.182000,1.219000,598.182000,13.219000
3,1,Libretón,459.986607,16.654410,498.904607,27.654410
4,1,Básico,501.236607,16.654410,530.397608,27.654410
...,...,...,...,...,...,...
2006,8,de,500.295960,773.127129,508.503960,782.127129
2007,8,la,510.555960,773.127129,516.297960,782.127129
2008,8,Ley,518.349960,773.127129,530.247960,782.127129
2009,8,Personas,532.299960,773.127129,563.475960,782.127129


In [3]:
bank = bank_detector.detect_bank()
statement_type = bank_detector.detect_statement_type()
statement_properties = bank_detector.get_statement_properties()

print(f'{bank} - {statement_type}')

bbva - debit


In [4]:
from table_boundary_detector import TransactionTableBoundaryDetector

boundary_detector = TransactionTableBoundaryDetector(bank_detector)

start_idx = boundary_detector.start_idx
end_idx = boundary_detector.end_idx

if statement_type == 'credit' and (start_idx is None or end_idx is None):
    print('New format')
    bank_detector = DefaultBankDetector(PDFReader(statement_path), new_credit_format=True)
    boundary_detector = TransactionTableBoundaryDetector(bank_detector)
    
    start_idx = boundary_detector.start_idx
    end_idx = boundary_detector.end_idx

df_table = boundary_detector.get_filtered_table_words()

print(f'{start_idx} - {end_idx}')
df_table

187 - 886


,page,text,x0,top,x1,bottom
0,1,FECHA,39.098979,516.402066,67.328979,526.402066
1,1,SALDO,518.600629,516.402066,547.920629,526.402066
2,1,OPER,21.333979,526.402066,43.843979,536.402066
3,1,LIQ,70.213979,526.402066,84.213979,536.402066
4,1,DESCRIPCION,104.588979,526.402066,161.648979,536.402066
...,...,...,...,...,...,...
694,4,"120,374.33",238.720628,355.996766,279.760628,365.996766
695,4,TOTAL,316.760628,355.996766,343.190628,365.996766
696,4,MOVIMIENTOS,345.470628,355.996766,403.790628,365.996766
697,4,ABONOS,406.070628,355.996766,441.160628,365.996766


In [5]:
from row_segmenter import TransactionRowSegmenter

row_segmenter = TransactionRowSegmenter(df_table, bank_detector)

column_delimitation = row_segmenter.delimit_column_positions()
grouped_rows = row_segmenter.group_rows()

print(row_segmenter.row_threshold)
print(column_delimitation)
grouped_rows

6.4865683568627475
{'columns': ['OPER', 'LIQ', 'DESCRIPCION', 'REFERENCIA', 'CARGOS', 'ABONOS', 'OPERACION', 'LIQUIDACION'], 'x0': [21.333978999999886, 70.21397899999988, 104.58897899999988, 321.1636089999999, 381.0512709999999, 423.43566899999985, 463.42397599999987, 536.2889759999998], 'x1': [43.84397899999989, 84.21397899999988, 161.64897899999988, 372.4436089999999, 416.63127099999986, 460.26566899999983, 513.7139759999999, 593.5889759999999]}


,row_group,text,words,top,bottom,page
0,0,FECHA SALDO,"[(FECHA, 39.098978999999886, 67.32897899999989...",516.402066,526.402066,1
1,1,OPER LIQ DESCRIPCION REFERENCIA CARGOS ABONOS ...,"[(OPER, 21.333978999999886, 43.84397899999989)...",526.402066,536.402066,1
2,2,15/MAY 16/MAY PAGO CUENTA DE TERCERO 40.00,"[(15/MAY, 14.803975999999807, 50.3739759999998...",543.710649,553.710649,1
3,3,Referencia 1634723960 BNET 1592044105 PAGO PRE...,"[(Referencia, 318.5299959999998, 364.462495999...",555.469940,564.969940,1
4,4,15/MAY 16/MAY SPEI ENVIADO SANTANDER 25.00,"[(15/MAY, 14.803975999999864, 50.3739759999998...",566.229231,576.229231,1
...,...,...,...,...,...,...
148,148,MBAN01002206130060313649,"[(MBAN01002206130060313649, 106.17898799999998...",289.062987,299.062987,4
149,149,CETES DIRECTO,"[(CETES, 106.17898799999998, 139.5189879999999...",300.242987,310.242987,4
150,150,Total de Movimientos,"[(Total, 8.588987999999972, 33.72898799999997)...",317.866924,329.866924,4
151,151,"TOTAL IMPORTE CARGOS 318,672.43 TOTAL MOVIMIEN...","[(TOTAL, 25.588987999999972, 52.01898799999997...",341.823538,351.823538,4


In [6]:
from table_reconstructor import TransactionTableReconstructor

table_reconstructor = TransactionTableReconstructor(grouped_rows, column_delimitation, bank_detector)

table_reconstructor.get_structured_table()

,OPER,LIQ,DESCRIPCION,REFERENCIA,CARGOS,ABONOS,OPERACION,LIQUIDACION
0,FECHA,,,,,,,
1,OPER,LIQ,DESCRIPCION CARGOS ABONOS LIQUIDACION,REFERENCIA,,,OPERACION,
2,15/MAY,,PAGO CUENTA DE TERCERO,,40.00,,,
3,,,BNET 1592044105 PAGO PRESTAMO,Referencia 1634723960,,,,
4,15/MAY,,SPEI ENVIADO SANTANDER,,25.00,,,
...,...,...,...,...,...,...,...,...
148,,,MBAN01002206130060313649,,,,,
149,,,CETES DIRECTO,,,,,
150,Total de,,,,,,,
151,TOTAL,,"CARGOS 318,672.43 CARGOS 30",TOTAL MOVIMIENTOS,,,,


In [7]:
df_structured = table_reconstructor.reconstruct_table()
df_structured

,OPER,LIQ,DESCRIPCION,REFERENCIA,CARGOS,ABONOS,OPERACION,LIQUIDACION
0,15/MAY,,PAGO CUENTA DE TERCERO,,40.00,,,
1,15/MAY,,SPEI ENVIADO SANTANDER,,25.00,,,
2,15/MAY,,PAGO CUENTA DE TERCERO,,60.00,,"266,326.99","266,451.99"
3,17/MAY,,SPEI ENVIADO SANTANDER,,850.00,,"265,476.99","265,476.99"
4,18/MAY,,SPEI ENVIADO STP,,"80,000.00",,,
5,18/MAY,,SPEI ENVIADO STP,,500.00,,,
6,18/MAY,,SAT,,"13,621.00",,,
7,18/MAY,,SAT,,"7,398.00",,,
8,18/MAY,,SAT,,"3,832.00",,,
9,18/MAY,,SAT,,"4,971.00",,,


In [8]:
from table_normalizer import TransactionTableNormalizer

normalizer = TransactionTableNormalizer(df_structured, bank_detector)

df_normalized = normalizer.normalize_table()
df_normalized

,Date,Description,Amount,Type
0,2022-05-15,PAGO CUENTA DE TERCERO,-40.00,Cargo
2,2022-05-15,PAGO CUENTA DE TERCERO,-60.00,Cargo
36,2022-05-15,Saldo Inicial,266451.99,Saldo Inicial
1,2022-05-15,SPEI ENVIADO SANTANDER,-25.00,Cargo
3,2022-05-17,SPEI ENVIADO SANTANDER,-850.00,Cargo
4,2022-05-18,SPEI ENVIADO STP,-80000.00,Cargo
5,2022-05-18,SPEI ENVIADO STP,-500.00,Cargo
6,2022-05-18,SAT,-13621.00,Cargo
7,2022-05-18,SAT,-7398.00,Cargo
8,2022-05-18,SAT,-3832.00,Cargo


In [9]:
normalizer.period_idx

12

In [10]:
from data_exporter import CsvExporter
from config import OUTPUTS_FOLDER

exporter = CsvExporter(df_normalized)
file_path = os.path.join(OUTPUTS_FOLDER, bank, f'{bank}_{statement_type}.csv')

exporter.export_to_csv(file_path)

Data exported to c:\Users\mario\Proyectos\Aether\Aether_v1\documents\outputs\bbva\bbva_debit.csv successfully.
